# Experimental fluxomics nullspace check

In [1]:
# Automatic module reloading
%load_ext autoreload
%autoreload 2

# Packages
import os
import sys
import pandas as pd 
import cobra
import numpy as np

# Directories
ROOT_DIR = os.path.abspath('..')

if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)
    
experimental_data_dir = os.path.join(ROOT_DIR, "data", "experimental", "fluxes_W3110_MD121_M9+Glu_none.csv")

## 1. Using all experimental reactions

In [2]:
model_xml = os.path.join(ROOT_DIR, "models", f"iML1515_GEM.xml")

# Create a copy to work on
model = cobra.io.read_sbml_model(model_xml)
experimental_data = pd.read_csv(experimental_data_dir)

# Convert DataFrame to dictionary where rxn_id is key and exp_flux is value
experimental_flux_dict = dict(zip(experimental_data['rxn_id'], experimental_data['exp_flux']))
print(f"Dictionary length: {len(experimental_flux_dict)}")

Dictionary length: 143


**Biomass fix**

The fluxomics study used the wild-type strain E. coli K-12 MG1655, so the biomass growth rate value will only be set to this reaction: BIOMASS_Ec_iML1515_WT_75p37M

In [3]:
# Delete the core reaction from the dictionary
del experimental_flux_dict['BIOMASS_Ec_iML1515_core_75p37M']
print(f"Dictionary length: {len(experimental_flux_dict)}")

Dictionary length: 142


**Filter transport reactions**

The mapped fluxomics data currently assigns the flux value of exchange reactions (EX_) to transport reactions ('tpp', 'tex', 'rpp'). This should be discarded when using experimental data to constrain a model, because the measured flux corresponds to just the exchange reaction. 

DOUBT: Is it valid to keep for transport reactions when doing a GEM vs experimental flux comparison?

In [4]:
transport_suffixes = ('tpp', 'tex', 'rpp')

print(f"Dict before removing transport reactions: {len(experimental_flux_dict)}")

experimental_flux_dict = {
    rxn_id: data for rxn_id, data in experimental_flux_dict.items()
    if not rxn_id.endswith(transport_suffixes)
}

print(f"Dict after removing transport reactions: {len(experimental_flux_dict)}")

Dict before removing transport reactions: 142
Dict after removing transport reactions: 132


**Apply experimental fluxes as bounds**

TO DO: Get the LB/UB from the Crown SI Excel ([1,2]Glc column)
    * Will have to ask Krishna how to treat these bounds. Some of them 
    are reported as (0,Inf).

In [5]:
from kinGEMs.nullspace_utils import apply_experimental_fluxes

constrained_model = apply_experimental_fluxes(model, 
                                              experimental_flux_dict, 
                                              stdev_factor=0.02,
                                              verbose=True)

c:\Users\lyachinas\miniconda3\envs\kingems\Lib\site-packages\bioservices\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2025-11-10 14:35:02.820 | INFO     | kinGEMs.config:<module>:12 - PROJ_ROOT path is: C:\Users\lyachinas\OneDrive - University of Toronto\GitHub\kinGEMs_v2


SHK3Dr: Kept experimental lower bound 0.1274 (orig: -1000.0)
SHK3Dr: Kept experimental upper bound 0.1326 (orig: 1000.0)

G5SD: Kept experimental lower bound 0.1568 (orig: 0.0)
G5SD: Kept experimental upper bound 0.1632 (orig: 1000.0)

CS: Kept experimental lower bound 2.0873999999999997 (orig: 0.0)
CS: Kept experimental upper bound 2.1726 (orig: 1000.0)

ICDHyr: Kept experimental lower bound 1.9796 (orig: -1000.0)
ICDHyr: Kept experimental upper bound 2.0604 (orig: 1000.0)

PPCK: Kept experimental lower bound 0.0294 (orig: 0.0)
PPCK: Kept experimental upper bound 0.0306 (orig: 1000.0)

ME1: Kept experimental lower bound 0.0294 (orig: 0.0)
ME1: Kept experimental upper bound 0.0306 (orig: 1000.0)

ALATA_L: Kept experimental lower bound -0.3774 (orig: -1000.0)
ALATA_L: Kept experimental upper bound -0.3626 (orig: 1000.0)

ASPTA: Kept experimental lower bound -1.6218000000000001 (orig: -1000.0)
ASPTA: Kept experimental upper bound -1.5582 (orig: 1000.0)

PYK: Kept experimental lower bound

In [6]:
from kinGEMs.nullspace_utils import optimize_model

solution_vector = optimize_model(constrained_model)

FAILURE: The experimental data is INCONSISTENT with the model.


In [7]:
from kinGEMs.nullspace_utils import find_infeasible_constraints

infeasible_constraints = find_infeasible_constraints(constrained_model)

print("Metabolites causing inconsistency:")
for constraint in infeasible_constraints:
    print(constraint.name)

Metabolites causing inconsistency:
13dpg_c
mocogdp_c
frulysp_c
progly_c
clpn161_p
citr__L_c
thmpp_c
murein5p5p_p
adocbl_c
xu5p__D_c
aspsa_c
fadh2_c
gp4g_c
cgly_c
3pg_c
ctp_c
xu5p__L_c
murein4px4p4p_p
phpyr_c
argsuc_c
chor_c
hom__L_c
4hbz_c
pe160_p
adpglc_c
clpn160_p
clpn181_p
ru5p__D_c
man6pglyc_c
colipa_e
pe181_p
3ig3p_c
dctp_c
utp_c
ahcys_c
uaagmda_c
3mob_c
hemeO_c
gmhep7p_c
2dmmq8_c
gtspmd_c
uppg3_c
mococdp_c
mlthf_c
2ippm_c
3psme_c
murein5p5p5p_p
fmn_c
q8_c
pphn_c
ap5a_c
pheme_c
so4_c
so4_p
pe161_p
rephaccoa_c
2fe2s_c
btn_c
malthp_c
nadph_c
4fe4s_c
icit_c
no3_c
dgtp_c
lipoamp_c
enter_c
seramp_c
5fthf_c


In [8]:
from kinGEMs.nullspace_utils import identify_reactions_causing_imbalance

problematic_reactions = identify_reactions_causing_imbalance(
    constrained_model, 
    experimental_flux_dict, 
    infeasible_constraints
)


Metabolite: 13dpg_c
Constrained reactions involving this metabolite:
  GAPD: coeff=+1.00, flux=17.1300, net=+17.1300
    g3p_c + nad_c + pi_c --> 13dpg_c + h_c + nadh_c
  PGK: coeff=+1.00, flux=-0.8100, net=-0.8100
    3pg_c + atp_c <-- 13dpg_c + adp_c

Metabolite: mocogdp_c
Constrained reactions involving this metabolite:
  BIOMASS_Ec_iML1515_WT_75p37M: coeff=-0.00, flux=0.7500, net=-0.0000
    0.000223 10fthf_c + 0.000223 2dmmql8_c + 2.5e-05 2fe2s_c + 0.000248 4fe4s_c + 0.000223 5mthf_c + 0.000279 accoa_c + 0.000223 adocbl_c + 0.499149 ala__L_c + 0.000223 amet_c + 0.28742 arg__L_c + 0.234232 asn__L_c + 0.234232 asp__L_c + 75.55223 atp_c + 2e-06 btn_c + 0.004952 ca2_c + 0.000223 chor_c + 0.004952 cl_c + 0.002944 clpn160_p + 0.00229 clpn161_p + 0.00118 clpn181_p + 0.000168 coa_c + 2.4e-05 cobalt2_c + 0.008151 colipa_e + 0.129799 ctp_c + 0.000674 cu2_c + 0.088988 cys__L_c + 0.024805 datp_c + 0.025612 dctp_c + 0.025612 dgtp_c + 0.024805 dttp_c + 0.000223 enter_c + 0.000223 fad_c + 0.006

## 2. Using JUST medium reactions (exchanges)

In [10]:
model_xml = os.path.join(ROOT_DIR, "models", f"iML1515_GEM.xml")

# Create a copy to work on
model = cobra.io.read_sbml_model(model_xml)

In [11]:
medium_flux_dict = {
    "EX_glc__D_e": -10,
    "EX_so4_e": -0.175297,
    "EX_o2_e": -14.2804,
    "EX_nh4_e": -5.23859,
}

medium_bounds = {
    "EX_glc__D_e": (-10.02, -9.98),
    "EX_so4_e": (-0.19, -0.16),
    "EX_o2_e": (-14.98, -13.62),
    "EX_nh4_e": (-5.62, -4.84),
    #"EX_h2o_e": (-6.96, 6.96)
}

In [12]:
from kinGEMs.nullspace_utils import apply_experimental_fluxes

constrained_model = apply_experimental_fluxes(model, 
                                              medium_flux_dict, 
                                              experimental_bounds=medium_bounds)

EX_glc__D_e: Kept original lower bound -10.0 (exp: -10.02)
EX_glc__D_e: Kept experimental upper bound -9.98 (orig: 1000.0)

EX_so4_e: Kept experimental lower bound -0.19 (orig: -1000.0)
EX_so4_e: Kept experimental upper bound -0.16 (orig: 1000.0)

EX_o2_e: Kept experimental lower bound -14.98 (orig: -1000.0)
EX_o2_e: Kept experimental upper bound -13.62 (orig: 1000.0)

EX_nh4_e: Kept experimental lower bound -5.62 (orig: -1000.0)
EX_nh4_e: Kept experimental upper bound -4.84 (orig: 1000.0)



In [13]:
from kinGEMs.nullspace_utils import optimize_model

solution_vector = optimize_model(constrained_model, fluxes_to_print=medium_flux_dict)

SUCCESS: The experimental data is CONSISTENT with the model.
             fluxes
EX_glc__D_e  -10.00
EX_so4_e      -0.16
EX_o2_e      -14.98
EX_nh4_e      -5.62


# Takeaways

- The experimental mapping needs correction
    - Back when I did these mappings I assigned to multiple iML1515 reactions the same flux value of a single measured experimental reaction - because this single reaction represented a cluster of multiple reactions in iML1515.
    - To address this:
        1. Go back to the interim dataframes in the fluxomics mapping (BactoCat project) and check which cluster reactions I expanded against `problematic_reactions` - remove these
        2. Re-run pipeline. There should be no problem (hopefully)
            - If there are still problematic  `problematic_reactions`: how to address these?
    - These corrections are necessary for a flux-to-flux comparison on many datapoints. A way around this would be to keep just the very few reactions that aren't lumped (about 20).

- All media bounds are properly mapped
    - Solution is feasible


# Remapping experimental fluxes

Remapped the experimental fluxes to their GEM counterpart. 

This was done using NotebookLM, a scv with the iML1515 reactions, a csv with the experimental reactions, and prompting 'map the entries in  'EXP REACTIONS' using exp_rxn_id, to its counterpart in 'GEM REACTIONS''. 

Furthermore, the data used for fluxomics corresponds to the 'COMPLETE-MFA' column in the Crown study SI Excel. This combines all 14 data sets (each corresponding to an isotopic tracer) to produce a single, unified flux map. This approach resolved more fluxes with smaller confidence intervals than any single tracer could.

The csv with the mapping and COMPLETE-MFA fluxes (with CIs) is under /data/experimental/crown_fluxomics_mapped.csv

The final csv to use is under /data/experimental/crown_fluxomics_final.csv. This was:
- cleaned to keep a single exp_rxn mapped to its gem_rxn
- all fluxes were divided by a factor of 10 (to match GEM 10 mmol glucose uptake)

In [78]:
model_xml = os.path.join(ROOT_DIR, "models", f"iML1515_GEM.xml")

# Create a copy to work on
model = cobra.io.read_sbml_model(model_xml)

In [79]:
final_experimental_data_dir = os.path.join(ROOT_DIR, "data", "experimental", "crown_fluxomics_final.csv")
experimental_data = pd.read_csv(final_experimental_data_dir)

# Convert DataFrame to dictionary where rxn_id is key and exp_flux is value
experimental_flux_dict = dict(zip(experimental_data['rxn_id'], experimental_data['exp_flux']))
print(f"Dictionary length: {len(experimental_flux_dict)}")

Dictionary length: 46


In [80]:
bounds_tuples = zip(
    experimental_data['exp_flux_lb'],
    experimental_data['exp_flux_ub'], 
)

# Zip the reaction IDs with the flux tuples
experimental_bounds_dict = dict(zip(experimental_data['rxn_id'], bounds_tuples))

In [81]:
from kinGEMs.nullspace_utils import apply_experimental_fluxes

constrained_model = apply_experimental_fluxes(model, 
                                              experimental_flux_dict, 
                                              verbose=True, 
                                              experimental_bounds=experimental_bounds_dict)

GLCptspp: Kept experimental lower bound 9.98049 (orig: 0.0)
GLCptspp: Kept experimental upper bound 10.0195 (orig: 1000.0)

PGI: Kept experimental lower bound 7.05523 (orig: -1000.0)
PGI: Kept experimental upper bound 7.26009 (orig: 1000.0)

PFK: Kept experimental lower bound 8.20648 (orig: 0.0)
PFK: Kept experimental upper bound 8.32198 (orig: 1000.0)

FBA: Kept experimental lower bound 8.20648 (orig: -1000.0)
FBA: Kept experimental upper bound 8.32198 (orig: 1000.0)

TPI: Kept experimental lower bound 8.20648 (orig: -1000.0)
TPI: Kept experimental upper bound 8.32198 (orig: 1000.0)

PYK: Kept experimental lower bound 2.28389 (orig: 0.0)
PYK: Kept experimental upper bound 3.14417 (orig: 1000.0)

G6PDH2r: Kept experimental lower bound 2.58343 (orig: -1000.0)
G6PDH2r: Kept experimental upper bound 2.79442 (orig: 1000.0)

GND: Kept experimental lower bound 2.44507 (orig: 0.0)
GND: Kept experimental upper bound 2.65122 (orig: 1000.0)

RPE: Kept experimental lower bound 1.06981 (orig: -100

In [82]:
from kinGEMs.nullspace_utils import optimize_model

solution_vector = optimize_model(constrained_model)

FAILURE: The experimental data is INCONSISTENT with the model.


In [83]:
from kinGEMs.nullspace_utils import find_infeasible_constraints

infeasible_constraints = find_infeasible_constraints(constrained_model)

print("Metabolites causing inconsistency:")
for constraint in infeasible_constraints:
    print(constraint.name)

Metabolites causing inconsistency:
pydx5p_c
thmpp_c
murein5p5p_p
adocbl_c
xu5p__D_c
5apru_c
sbzcoa_c
tyr__L_c
phthr_c
colipa_e
3ig3p_c
ahcys_c
uaagmda_c
ile__L_c
xmp_c
2omhmbl_c
hemeO_c
2cpr5p_c
gtspmd_c
uppg3_c
phe__L_c
malcoa_c
phom_c
murein5p5p5p_p
ap5a_c
pheme_c
dnad_c
25aics_c
enter_c
4pasp_c
btnso_c
4abzglu_c


In [84]:
from kinGEMs.nullspace_utils import identify_reactions_causing_imbalance

problematic_reactions = identify_reactions_causing_imbalance(
    constrained_model, 
    experimental_flux_dict, 
    infeasible_constraints
)


Metabolite: pydx5p_c
Constrained reactions involving this metabolite:
  BIOMASS_Ec_iML1515_WT_75p37M: coeff=-0.00, flux=0.7523, net=-0.0002
    0.000223 10fthf_c + 0.000223 2dmmql8_c + 2.5e-05 2fe2s_c + 0.000248 4fe4s_c + 0.000223 5mthf_c + 0.000279 accoa_c + 0.000223 adocbl_c + 0.499149 ala__L_c + 0.000223 amet_c + 0.28742 arg__L_c + 0.234232 asn__L_c + 0.234232 asp__L_c + 75.55223 atp_c + 2e-06 btn_c + 0.004952 ca2_c + 0.000223 chor_c + 0.004952 cl_c + 0.002944 clpn160_p + 0.00229 clpn161_p + 0.00118 clpn181_p + 0.000168 coa_c + 2.4e-05 cobalt2_c + 0.008151 colipa_e + 0.129799 ctp_c + 0.000674 cu2_c + 0.088988 cys__L_c + 0.024805 datp_c + 0.025612 dctp_c + 0.025612 dgtp_c + 0.024805 dttp_c + 0.000223 enter_c + 0.000223 fad_c + 0.006388 fe2_c + 0.007428 fe3_c + 0.255712 gln__L_c + 0.255712 glu__L_c + 0.595297 gly_c + 0.154187 glycogen_c + 0.000223 gthrd_c + 0.209121 gtp_c + 70.028756 h2o_c + 0.000223 hemeO_c + 0.092056 his__L_c + 0.282306 ile__L_c + 0.18569 k_c + 0.437778 leu__L_c + 

## Removing problematic reactions

From the print above: TKT2, TKT1, RPE, SUCDi, ASNS2

In [85]:
model_xml = os.path.join(ROOT_DIR, "models", f"iML1515_GEM.xml")

# Create a copy to work on
model = cobra.io.read_sbml_model(model_xml)

In [86]:
final_experimental_data_dir = os.path.join(ROOT_DIR, "data", "experimental", "crown_fluxomics_final.csv")
experimental_data = pd.read_csv(final_experimental_data_dir)

# Convert DataFrame to dictionary where rxn_id is key and exp_flux is value
experimental_flux_dict = dict(zip(experimental_data['rxn_id'], experimental_data['exp_flux']))
print(f"Dictionary length: {len(experimental_flux_dict)}")


Dictionary length: 46


In [87]:
keys_to_remove = ['TKT1', 'TKT2', 'RPE', 'ASNS2', 'THRD']

print(f"Original dictionary size: {len(experimental_flux_dict)}")

for key in keys_to_remove:
    del experimental_flux_dict[key]

print(f"Dictionary length: {len(experimental_flux_dict)}")

Original dictionary size: 46
Dictionary length: 41


In [88]:
from kinGEMs.nullspace_utils import apply_experimental_fluxes

constrained_model = apply_experimental_fluxes(model, 
                                              experimental_flux_dict, 
                                              verbose=True, 
                                              experimental_bounds=experimental_bounds_dict)

GLCptspp: Kept experimental lower bound 9.98049 (orig: 0.0)
GLCptspp: Kept experimental upper bound 10.0195 (orig: 1000.0)

PGI: Kept experimental lower bound 7.05523 (orig: -1000.0)
PGI: Kept experimental upper bound 7.26009 (orig: 1000.0)

PFK: Kept experimental lower bound 8.20648 (orig: 0.0)
PFK: Kept experimental upper bound 8.32198 (orig: 1000.0)

FBA: Kept experimental lower bound 8.20648 (orig: -1000.0)
FBA: Kept experimental upper bound 8.32198 (orig: 1000.0)

TPI: Kept experimental lower bound 8.20648 (orig: -1000.0)
TPI: Kept experimental upper bound 8.32198 (orig: 1000.0)

PYK: Kept experimental lower bound 2.28389 (orig: 0.0)
PYK: Kept experimental upper bound 3.14417 (orig: 1000.0)

G6PDH2r: Kept experimental lower bound 2.58343 (orig: -1000.0)
G6PDH2r: Kept experimental upper bound 2.79442 (orig: 1000.0)

GND: Kept experimental lower bound 2.44507 (orig: 0.0)
GND: Kept experimental upper bound 2.65122 (orig: 1000.0)

RPI: Kept experimental lower bound 1.34422 (orig: -100

In [89]:
from kinGEMs.nullspace_utils import optimize_model

solution_vector = optimize_model(constrained_model)

FAILURE: The experimental data is INCONSISTENT with the model.


In [90]:
from kinGEMs.nullspace_utils import find_infeasible_constraints

infeasible_constraints = find_infeasible_constraints(constrained_model)

print("Metabolites causing inconsistency:")
for constraint in infeasible_constraints:
    print(constraint.name)

Metabolites causing inconsistency:
cdpdhdec9eg_c
thmpp_c
adocbl_c
sheme_c
2p4c2me_c
amet_c
sbzcoa_c
3ig3p_c
ahcys_c
3ohodcoa_c
ile__L_c
gln__L_c
xmp_c
hemeO_c
ckdo_c
gtspmd_c
mococdp_c
cdpdodec11eg_c
phom_c
murein5p5p5p_p
fmn_c
ap5a_c
pheme_c
dnad_c
25aics_c
fad_c
cdpdhdecg_c
4pasp_c
btnso_c
5fthf_c


In [91]:
from kinGEMs.nullspace_utils import identify_reactions_causing_imbalance

problematic_reactions = identify_reactions_causing_imbalance(
    constrained_model, 
    experimental_flux_dict, 
    infeasible_constraints
)


Metabolite: thmpp_c
Constrained reactions involving this metabolite:
  BIOMASS_Ec_iML1515_WT_75p37M: coeff=-0.00, flux=0.7523, net=-0.0002
    0.000223 10fthf_c + 0.000223 2dmmql8_c + 2.5e-05 2fe2s_c + 0.000248 4fe4s_c + 0.000223 5mthf_c + 0.000279 accoa_c + 0.000223 adocbl_c + 0.499149 ala__L_c + 0.000223 amet_c + 0.28742 arg__L_c + 0.234232 asn__L_c + 0.234232 asp__L_c + 75.55223 atp_c + 2e-06 btn_c + 0.004952 ca2_c + 0.000223 chor_c + 0.004952 cl_c + 0.002944 clpn160_p + 0.00229 clpn161_p + 0.00118 clpn181_p + 0.000168 coa_c + 2.4e-05 cobalt2_c + 0.008151 colipa_e + 0.129799 ctp_c + 0.000674 cu2_c + 0.088988 cys__L_c + 0.024805 datp_c + 0.025612 dctp_c + 0.025612 dgtp_c + 0.024805 dttp_c + 0.000223 enter_c + 0.000223 fad_c + 0.006388 fe2_c + 0.007428 fe3_c + 0.255712 gln__L_c + 0.255712 glu__L_c + 0.595297 gly_c + 0.154187 glycogen_c + 0.000223 gthrd_c + 0.209121 gtp_c + 70.028756 h2o_c + 0.000223 hemeO_c + 0.092056 his__L_c + 0.282306 ile__L_c + 0.18569 k_c + 0.437778 leu__L_c + 3

## Keeping just reactions from ECOMICS

TO DO